# 🪰 RAPID v3 — Fly-CL on CIFAR-100
> **Block-wise Absolute WTA · NCM Prototype Classifier · Fisher-adaptive Projection**

| Param | Value |
|---|---|
| Dataset | CIFAR-100 (100 classes / 10 tasks) |
| Backbone | ViT-B/16 (frozen) |
| Projection | Growable Sparse Random (expand_dim=10000) |
| Classifier | NCM + Per-class Subspace Cosine |
| Normalization | [-1, 1] (Paper Appendix C.3) |

---
Run all cells top-to-bottom. Results appear in **Cell 6** and **Cell 7**.

## ⚙️ Step 1 — Install dependencies

In [ ]:
# timm: ViT backbone loader
!pip install timm==0.9.16 -q
print("✅ Dependencies ready.")

## 📦 Step 2 — Clone / update repo

In [ ]:
import os

REPO_URL = "https://github.com/ZaPhat206/LAB_FLY.git"
WORK_DIR = "/kaggle/working/LAB_FLY"

print("Cleaning up old repo...")
!rm -rf {WORK_DIR}

print("Cloning fresh repo...")
!git clone {REPO_URL} {WORK_DIR} -q

os.chdir(WORK_DIR)
print(f"✅ Working dir: {os.getcwd()}")
print("   Latest Commit:")
!git log -1 --format="      %h | %cd | %s"


## 🧠 Step 3 — Download pretrained ViT-B/16

In [ ]:
import os
os.chdir(f"{WORK_DIR}/pretrained_model")
!sh download.sh
os.chdir(WORK_DIR)
print("✅ Pretrained model ready.")

## 🔧 Step 4 — Fix Kaggle package conflict

In [ ]:
# Kaggle pre-installs HuggingFace 'datasets' → rename our folder to avoid import conflict
import os, shutil
os.chdir(WORK_DIR)

if os.path.isdir("datasets") and not os.path.isdir("my_datasets"):
    shutil.move("datasets", "my_datasets")
    print("Renamed: datasets → my_datasets")
else:
    print("Rename: already done or not needed.")

!sed -i 's/from datasets.load_dataset/from my_datasets.load_dataset/g' main.py
!find my_datasets -name '*.py' -exec sed -i 's/from datasets/from my_datasets/g' {} + 2>/dev/null || true
print("✅ Import conflict fixed.")

## 📂 Step 5 — Link CIFAR-100 dataset
> **Prerequisite:** Add your uploaded CIFAR-100 dataset via the **Data tab** (right panel).  
> The cell below auto-detects its path — no internet download needed.

In [ ]:
import os, glob, builtins

# Auto-detect CIFAR-100 inside any added Kaggle dataset
# torchvision expects: root/cifar-100-python/{meta, train, test}
matches = glob.glob("/kaggle/input/*/cifar-100-python")

if matches:
    CIFAR_PYTHON_DIR = matches[0]
    CIFAR_ROOT       = os.path.dirname(CIFAR_PYTHON_DIR)
    builtins.CIFAR_ROOT = CIFAR_ROOT
    print(f"✅ CIFAR-100 found  : {CIFAR_PYTHON_DIR}")
    print(f"   Will pass --root : {CIFAR_ROOT}")
    # Quick sanity check
    for f in ["meta", "train", "test"]:
        path = os.path.join(CIFAR_PYTHON_DIR, f)
        status = "✅" if os.path.exists(path) else "❌ MISSING"
        print(f"   {status}  {path}")
else:
    print("⚠️  CIFAR-100 dataset NOT found in /kaggle/input/")
    print("   → Go to the Data tab → Add Data → search your dataset.")
    print("   → Falling back to auto-download (slow, ~161 MB).")
    builtins.CIFAR_ROOT = "../data"

## 🚀 Step 6 — Run RAPID v3

In [ ]:
# --- [ CELL 6 ] --- Lệnh chạy Training ---
import subprocess
import sys

cifar_root = "/kaggle/input/cifar-100/cifar-100-python"

CMD = [
    sys.executable, "main.py",
    "--dataset",           "CIFAR-100",
    "--root",              cifar_root,
    "--num_classes",       "100",
    "--num_tasks",         "10",
    "--model_name",        "vit_base_patch16_224",
    "--embedding_dim",     "768",
    "--expand_dim",        "10000",
    "--synaptic_degree",   "300",
    "--coding_level",      "0.01",
    
    # 🔥 Combo 1: Ridge + Subspace + Procrustes
    "--classifier_type",   "ridge_subspace",
    "--use_subspace", 
    "--use_procrustes",
    
    "--fisher_block",      "512",
    "--fisher_sat",        "0.005",
    "--data_augmentation", "vit",
    "--seed",              "1993",
    "--batch_size",        "128",
    "--gpu",               "0",
]

print("Running command:")
print(" ".join(CMD))

# Chạy sub-process và in log trực tiếp
process = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(process.stdout.readline, ''):
    print(line, end='')

process.wait()
